# Phase 1 — PaddleOCR (detect) + VietOCR (recognize)

Notebook demo từng bước pipeline nhận diện chữ viết tay tiếng Việt:

1. Load ảnh
2. Chạy **PaddleOCR detector** lấy polygon các vùng text
3. Vẽ bbox lên ảnh để kiểm tra
4. Crop từng vùng bằng `four_point_transform`
5. Chạy **VietOCR recognizer** trên batch crops
6. Ghép kết quả end-to-end qua `HandwritingOCRPipeline`

> Nếu chạy trên Colab, bỏ comment cell cài đặt bên dưới.

In [ ]:
# Colab setup (bỏ comment khi cần)
# !pip install -q paddlepaddle-gpu==2.6.1 paddleocr==2.7.3 vietocr==0.3.13
# !pip install -q "numpy<2.0" "opencv-python==4.10.0.84"
# !pip install -q streamlit pyyaml

In [ ]:
import sys, os
from pathlib import Path

# Thêm thư mục gốc repo vào sys.path để import `src`
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print('ROOT =', ROOT)

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

%matplotlib inline

## 1. Load ảnh mẫu

In [ ]:
IMG_PATH = ROOT / 'data' / 'samples' / 'note.jpg'

if not IMG_PATH.is_file():
    # Fallback: lấy ngẫu nhiên file ảnh đầu tiên trong data/samples
    candidates = sorted((ROOT / 'data' / 'samples').glob('*'))
    candidates = [p for p in candidates if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp'}]
    assert candidates, 'Hãy đặt ít nhất một ảnh vào data/samples/'
    IMG_PATH = candidates[0]

print('Image:', IMG_PATH)
bgr = cv2.imdecode(np.frombuffer(IMG_PATH.read_bytes(), np.uint8), cv2.IMREAD_COLOR)
rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(10, 6))
plt.imshow(rgb); plt.axis('off'); plt.title('Input'); plt.show()

## 2. PaddleOCR detector

In [ ]:
from src.detector import TextDetector

detector = TextDetector(lang='vi', use_angle_cls=True, use_gpu=False,
                        det_db_unclip_ratio=2.0)
polygons = detector.detect(bgr)
print(f'Detected {len(polygons)} boxes')

In [ ]:
from src.utils.viz import draw_ocr_result

items_only_box = [{'bbox': p, 'text': ''} for p in polygons]
viz = draw_ocr_result(bgr, items_only_box, show_text=False, line_width=3)
plt.figure(figsize=(10, 6))
plt.imshow(viz); plt.axis('off'); plt.title(f'{len(polygons)} detected regions'); plt.show()

## 3. Crop từng vùng + sắp xếp theo reading order

In [ ]:
from src.utils.image import (
    expand_polygon, four_point_transform, sort_polygons_reading_order,
)

order = sort_polygons_reading_order(polygons, y_tolerance=10)
polys_sorted = [polygons[i] for i in order]

crops = []
for p in polys_sorted:
    p_exp = expand_polygon(p, ratio=0.05)
    warped = four_point_transform(bgr, p_exp)
    crops.append(Image.fromarray(cv2.cvtColor(warped, cv2.COLOR_BGR2RGB)))

n = min(len(crops), 12)
cols = 2
rows = (n + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(12, 1.4 * rows + 1))
axes = np.atleast_2d(axes)
for i in range(rows * cols):
    ax = axes[i // cols, i % cols]
    if i < n:
        ax.imshow(crops[i]); ax.set_title(f'crop #{i}', fontsize=9)
    ax.axis('off')
plt.tight_layout(); plt.show()

## 4. VietOCR recognizer

In [ ]:
from src.recognizer import TextRecognizer

rec = TextRecognizer(model='vgg_transformer', device='auto')
results = rec.recognize_batch(crops)
for i, (text, prob) in enumerate(results[:20]):
    print(f'#{i:02d} ({prob:.2f}) {text}')

## 5. End-to-end pipeline

In [ ]:
from src.pipeline import HandwritingOCRPipeline

pipe = HandwritingOCRPipeline.from_config(str(ROOT / 'configs' / 'default.yaml'))
result = pipe.run(str(IMG_PATH), save_visualization=True,
                  output_dir=str(ROOT / 'data' / 'outputs'))
print('num_boxes =', result['num_boxes'])
print('--- full_text ---')
print(result['full_text'])

In [ ]:
viz_path = result.get('visualization_path')
if viz_path:
    plt.figure(figsize=(12, 8))
    plt.imshow(Image.open(viz_path)); plt.axis('off'); plt.title('End-to-end result')
    plt.show()